# Install Keras Tuner

In [ ]:
!pip install keras-tuner

# Import Libraries and Load Data

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(x_train.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1532s 9us/step
(50000, 32, 32, 3)


# Define Model Building Function

In [ ]:
def build_model(hp):
  model = keras.Sequential([
      keras.layers.Conv2D(
          filters=hp.Int('conv_units', 32, 128, step=32),
          kernel_size=hp.Choice('kernel_size', [3,5]),
          activation='relu', input_shape=(32,32,3)
      ),
      keras.layers.MaxPooling2D(),
      keras.layers.Dropout(hp.Float('dropout', 0.1, 0.5, step=0.1)),
      keras.layers.Flatten(),
      keras.layers.Dense(hp.Int('dense_units', 64, 256, step=64), activation='relu'),
      keras.layers.Dense(10, activation='softmax')
  ])

  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=lr),
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy']
  )

  return model

# Initialize Keras Tuner

In [ ]:
tuner = kt.Hyperband(
    build_model, objective='val_accuracy', max_epochs=10,
    factor=3, directory='tuner_dir', project_name='lab'
)

# Search for Best Hyperparameters

In [ ]:
tuner.search(
    x_train, y_train, epochs=10, validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)]
)

Trial 30 Complete [00h 00m 31s]
val_accuracy: 0.10109999775886536

Best val_accuracy So Far: 0.6340000033378601
Total elapsed time: 00h 13m 17s


# Build and Train Model with Best Hyperparameters

In [ ]:
best_hps = tuner.get_best_hyperparameters(1)[0]
model = tuner.hypermodel.build(best_hps)
history = model.fit(x_train, y_train, epochs=10, validation_split=0.2, batch_size=64)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.1125 - loss: 3.8086 - val_accuracy: 0.1056 - val_loss: 2.2990
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.1569 - loss: 2.2215 - val_accuracy: 0.2200 - val_loss: 2.0432
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.2482 - loss: 1.9633 - val_accuracy: 0.3016 - val_loss: 1.7804
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3278 - loss: 1.7454 - val_accuracy: 0.3839 - val_loss: 1.6518
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4100 - loss: 1.5737 - val_accuracy: 0.4636 - val_loss: 1.4548
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4855 - loss: 1.4078 - val_accuracy: 0.5199 - val_loss: 1.3495
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5390 - loss: 1.2781 - val_accuracy: 0.5362 - val_loss: 1.3154
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5777 - loss: 1.1763 - val_accuracy: 0.

# Evaluate Model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Optimal Learning Rate: {best_hps.get('learning_rate')}")
print(f"Test Accuracy: {test_acc:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6068 - loss: 1.1287
Optimal Learning Rate: 0.0001
Test Accuracy: 0.6068
